> **Purpose:** Soft-Impute (NNM) vs IPW: p* scaling and runtime.  
> **Data:** synthesized

# Figure 11 — Nuclear Norm Minimization vs. IPW under uniform sampling

**Question (Candès & Recht / Candès & Plan):** under uniform Bernoulli(p) entry
sampling, does convex matrix completion via nuclear-norm minimization (NNM) allow
a smaller $p$ than the simple inverse-probability-weighted (IPW) estimator, for
the same Fiedler sign-recovery accuracy?  And how does the gap **scale with $n$**?

**Design (paired):** for each $(n, p, \text{rep})$ we draw one uniform mask
$\Omega$ and feed it to **both** estimators — eliminates sampling-luck variance.

- **IPW:** $\hat S_{ij} = M_{ij}/p$ on $\Omega$, $0$ off $\Omega$; diagonal preserved.
- **NNM:** Soft-Impute (Mazumder, Hastie, Tibshirani 2010), solving
  $\min_L \tfrac12 \|P_\Omega(M-L)\|_F^2 + \lambda\|L\|_\ast$.

**Synthetic similarity matrix:** balanced binary tree, $S_{ij} = \alpha^{d(i,j)}$
with $\alpha = 0.9$ (matching figure 1). Closed-form, deterministic — no
sequence simulation, no Kingman randomness. Reference Fiedler is the *exact*
population Fiedler $(+1\ldots-1)/\sqrt{n}$, perfectly balanced.
This is the **cleanest low-rank setting** for the comparison: Candès-Recht's
exact-rank-$r$ assumption is closest to matched here, so any NNM advantage
should be most visible.

## Cell 1 — Imports and configuration

In [ ]:
import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.utils import (
    uniform_mask, ipw_from_mask, nnm_from_mask,
    compute_fiedler_of_S, find_threshold_p_star,
)
from src.utils.metrics import compute_sign_agreement
from src.cache_io import notebook_dir

TREE_MODEL = "balanced_binary"
N_VALUES = [128, 512, 1024]
P_GRID = np.geomspace(1e-3, 1.0, 10)
B = 20
SEED_BASE = 0
RECOVERY_TARGETS = [0.90, 0.95]

# Outputs live under results/notebooks/, addressed through cache_io.notebook_dir --
# NOT beside the notebook. The previous path pointed at an
# analysis/03_sampling_methods/nnm_vs_ipw_outputs/ dir that
# was empty and has been removed; this notebook's actual cached trials/agg/thresholds
# CSVs have always been written to the scope below. The "03_sampling_methods" prefix is
# a historical key kept so those cached results still resolve -- it is a cache key, not
# a directory in this tree.
OUT_DIR = notebook_dir("03_sampling_methods/nnm_vs_ipw")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TRIALS_CSV = OUT_DIR / "trials.csv"
AGG_CSV    = OUT_DIR / "agg.csv"
THRESH_CSV = OUT_DIR / "thresholds.csv"

print(f"tree model: {TREE_MODEL}")
print(f"N values: {N_VALUES}")
print(f"P grid ({len(P_GRID)} points): {P_GRID.round(5).tolist()}")
print(f"B = {B} reps  →  total NNM solves = {len(N_VALUES) * len(P_GRID) * B}")


## Cell 2 — Build balanced-binary $S$ and reference Fiedler $v_{\text{pop}}$

For each $n$ build $S_{ij} = \alpha^{d(i,j)}$ from the closed form in
`utils/balanced_binary.py`. The population Fiedler is the exact
$\pm 1 / \sqrt{n}$ vector — no eigensolver needed, no random tree draw,
no cache lookup.

In [ ]:
from analysis.utils import (
    build_balanced_binary_S, balanced_binary_population_fiedler,
)

ALPHA = 0.9   # matches figure_1

M_by_n   = {}
fref_by_n = {}
for n in N_VALUES:
    t0 = time.perf_counter()
    M = build_balanced_binary_S(n, ALPHA, dtype=np.float64)
    f_pop = balanced_binary_population_fiedler(n)
    M_by_n[n] = M
    fref_by_n[n] = f_pop
    pos = int((f_pop > 0).sum())
    sv = np.linalg.svd(M, compute_uv=False)
    print(f"n={n:>5}:  build={time.perf_counter()-t0:>5.3f}s  "
          f"||M||_F={np.linalg.norm(M,'fro'):>7.2f}  "
          f"partition={pos}/{n}  (imbalance={max(pos,n-pos)/n:.3f})  "
          f"top-3 sv={sv[:3].round(2).tolist()}")


## Cell 3 — Paired sweep across $(n, p, \text{rep})$

In [ ]:
def recovery(f_hat, f_ref):
    s = compute_sign_agreement(f_ref, f_hat) / 100.0
    return max(s, 1.0 - s)

rows = []
total = len(N_VALUES) * len(P_GRID) * B
done = 0
t0 = time.perf_counter()
for n in N_VALUES:
    M = M_by_n[n]
    f_ref = fref_by_n[n]
    for p in P_GRID:
        p = float(p)
        # Degenerate-sample guard (matches utils/sweep.py:57): when expected
        # per-row samples p*n < 1, the Erdős-Rényi subgraph is subcritical and
        # the Fiedler computation on either reconstruction is unreliable. Floor
        # both methods at 0.5 (the random-coin floor before max(s,1-s)).
        # When degen=True we skip the expensive NNM solve entirely — it would
        # spend time on SVDs of a near-zero matrix only to have the result
        # discarded, and can itself fail to converge.
        degen = (p * n < 1.0)
        for rep in range(B):
            rng = np.random.default_rng(
                SEED_BASE * 10_000_000 + n * 10_000 + rep + int(round(p * 1e6))
            )
            Omega = uniform_mask(n, p, rng)

            t_ipw0 = time.perf_counter()
            S_ipw = ipw_from_mask(M, Omega, p)
            t_ipw_recon = time.perf_counter() - t_ipw0
            if degen:
                acc_ipw, t_ipw = 0.5, t_ipw_recon
            else:
                f_ipw = compute_fiedler_of_S(S_ipw, sampling_prob=p)
                t_ipw = time.perf_counter() - t_ipw0
                acc_ipw = recovery(f_ipw, f_ref)

            if degen:
                acc_nnm, t_nnm = 0.5, 0.0  # skip NNM solve entirely
            else:
                t_nnm0 = time.perf_counter()
                L_nnm = nnm_from_mask(M, Omega, max_iter=150, tol=1e-5)
                f_nnm = compute_fiedler_of_S(L_nnm)
                t_nnm = time.perf_counter() - t_nnm0
                acc_nnm = recovery(f_nnm, f_ref)

            rows.append({"n": n, "p": p, "rep": rep, "method": "ipw",
                         "recovery": acc_ipw, "runtime_s": t_ipw})
            rows.append({"n": n, "p": p, "rep": rep, "method": "nnm",
                         "recovery": acc_nnm, "runtime_s": t_nnm})
            done += 1
            if done % max(1, total // 30) == 0:
                el = time.perf_counter() - t0
                eta = el / done * (total - done)
                print(f"  {done}/{total}   elapsed {el:>5.0f}s   eta {eta:>5.0f}s")

df_trials = pd.DataFrame(rows)
df_trials.to_csv(TRIALS_CSV, index=False)
print(f"\nsaved {len(df_trials)} rows -> {TRIALS_CSV.name}")
print(f"total wall-clock: {time.perf_counter() - t0:.0f}s")

## Cell 4 — Aggregate per $(n, p, \text{method})$ and recover $p^*$

In [ ]:
df_agg = (
    df_trials.groupby(["n", "method", "p"])
             .agg(rec_mean=("recovery", "mean"),
                  rec_std=("recovery", "std"),
                  t_mean=("runtime_s", "mean"),
                  t_median=("runtime_s", "median"),
                  reps=("rep", "count"))
             .reset_index()
             .sort_values(["n", "method", "p"])
             .reset_index(drop=True)
)
df_agg.to_csv(AGG_CSV, index=False)

thresh_rows = []
for n in N_VALUES:
    for method in ["ipw", "nnm"]:
        sub = df_agg.query("n == @n and method == @method").sort_values("p")
        for tgt in RECOVERY_TARGETS:
            p_star = find_threshold_p_star(sub["p"].values, sub["rec_mean"].values, threshold=tgt)
            thresh_rows.append({"n": n, "method": method, "target": tgt, "p_star": p_star})
df_thresh = pd.DataFrame(thresh_rows)
df_thresh.to_csv(THRESH_CSV, index=False)

print(df_thresh.to_string(index=False))
print()
print("Sample-complexity savings of NNM (1 - p_nnm/p_ipw):")
for n in N_VALUES:
    for tgt in RECOVERY_TARGETS:
        p_ipw = df_thresh.query("n==@n and method=='ipw' and target==@tgt")["p_star"].iloc[0]
        p_nnm = df_thresh.query("n==@n and method=='nnm' and target==@tgt")["p_star"].iloc[0]
        if p_ipw and p_nnm:
            print(f"  n={n:>4}  tgt={tgt:.2f}:  p_ipw={p_ipw:.4f}  p_nnm={p_nnm:.4f}  "
                  f"savings={(1 - p_nnm/p_ipw)*100:>4.0f}%")

## Cell 5 — Accuracy curves, one panel per $n$

In [ ]:
METHOD_COLOR = {"ipw": "#1d4ed8", "nnm": "#b45309"}
METHOD_LABEL = {"ipw": "IPW", "nnm": "NNM (Soft-Impute)"}

fig, axes = plt.subplots(1, len(N_VALUES), figsize=(6 * len(N_VALUES), 7), sharey=True)
for ax, n in zip(axes, N_VALUES):
    for method in ["ipw", "nnm"]:
        sub = df_agg.query("n == @n and method == @method").sort_values("p")
        ax.plot(sub["p"], sub["rec_mean"], "-o", color=METHOD_COLOR[method],
                lw=2.0, markersize=5, label=METHOD_LABEL[method])
        ax.fill_between(sub["p"], sub["rec_mean"] - sub["rec_std"],
                        sub["rec_mean"] + sub["rec_std"],
                        color=METHOD_COLOR[method], alpha=0.18, linewidth=0)

    for tgt in RECOVERY_TARGETS:
        ax.axhline(tgt, color="0.6", lw=0.7, ls="--", zorder=0)

    p_ipw = df_thresh.query("n==@n and method=='ipw' and target==0.95")["p_star"].iloc[0]
    p_nnm = df_thresh.query("n==@n and method=='nnm' and target==0.95")["p_star"].iloc[0]
    if p_ipw is not None:
        ax.axvline(p_ipw, color=METHOD_COLOR["ipw"], lw=0.8, ls=":", alpha=0.7)
    if p_nnm is not None:
        ax.axvline(p_nnm, color=METHOD_COLOR["nnm"], lw=0.8, ls=":", alpha=0.7)

    label = (rf"$p^*_{{0.95}}$  NNM={p_nnm:.3f}  IPW={p_ipw:.3f}"
             if p_ipw and p_nnm else r"$p^*_{0.95}$ not reached")
    ax.text(0.02, 0.02, label, transform=ax.transAxes, fontsize=10, color="0.2",
            bbox=dict(boxstyle="round,pad=0.3", fc="#fff8e7", ec="0.7"))

    ax.axhline(0.5, color="0.4", lw=0.5, alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel(r"sampling rate $p$", fontsize=11)
    ax.set_title(f"n = {n}", fontsize=12)
    ax.set_ylim(0.45, 1.03)
    ax.grid(True, alpha=0.3)
    if n == N_VALUES[0]:
        ax.set_ylabel(r"sign-recovery  $\max(s, 1-s)$", fontsize=11)
    if n == N_VALUES[-1]:
        ax.legend(loc="lower right", fontsize=11, framealpha=0.95)

fig.suptitle(rf"NNM vs IPW under uniform sampling   (Kingman, $\mu={MU}$, $B={B}$)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "accuracy.png", dpi=140, bbox_inches="tight")
plt.show()

## Cell 6 — $p^*(n)$ scaling: the "actual impact" of NNM

Theory: Candès-Recht predicts $p^*_{\text{NNM}} \sim r \log^2 n / n$, while IPW's
threshold decays like $1/n$ at best — so the **gap should widen with $n$**.
Overlay the empirical thresholds with the theory curve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
n_arr = np.array(N_VALUES, dtype=float)
for method in ["ipw", "nnm"]:
    for tgt, ls, marker in zip(RECOVERY_TARGETS, ["--", "-"], ["s", "o"]):
        sub = df_thresh.query("method == @method and target == @tgt").sort_values("n")
        ps = sub["p_star"].values.astype(float)
        ns = sub["n"].values.astype(float)
        mask = np.isfinite(ps)
        ax.plot(ns[mask], ps[mask], ls, marker=marker, color=METHOD_COLOR[method],
                markersize=8, lw=2.0,
                label=f"{METHOD_LABEL[method]}  target={tgt:.2f}")

r = 2.0
theory_n = np.array(N_VALUES, dtype=float)
theory_p = r * np.log(theory_n)**2 / theory_n
ax.plot(theory_n, theory_p, ":", color="0.4", lw=1.5,
        label=r"$r \log^2 n / n$  (Candès-Recht, $r=2$)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"matrix size $n$", fontsize=12)
ax.set_ylabel(r"recovery threshold $p^*$", fontsize=12)
ax.set_title(r"Sample-complexity scaling with $n$", fontsize=12)
ax.grid(True, alpha=0.3, which="both")
ax.set_xticks(N_VALUES)
ax.set_xticklabels([str(n) for n in N_VALUES])
ax.legend(fontsize=10, framealpha=0.95)
plt.tight_layout()
plt.savefig(OUT_DIR / "scaling.png", dpi=140, bbox_inches="tight")
plt.show()

## Cell 7 — Runtime per reconstruction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
LS_BY_N = {N_VALUES[0]: "--", N_VALUES[1]: "-", N_VALUES[2]: "-"}
LW_BY_N = {N_VALUES[0]: 1.5,  N_VALUES[1]: 2.0, N_VALUES[2]: 3.0}
for method in ["ipw", "nnm"]:
    for n in N_VALUES:
        sub = df_agg.query("n == @n and method == @method").sort_values("p")
        ax.plot(sub["p"], sub["t_median"], LS_BY_N[n], color=METHOD_COLOR[method],
                lw=LW_BY_N[n], marker="o", markersize=4,
                label=f"{METHOD_LABEL[method]}, n={n}")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"sampling rate $p$", fontsize=12)
ax.set_ylabel("median wall-clock per reconstruction  [s]", fontsize=12)
ax.set_title(r"Per-reconstruction cost  (NNM scales as $\mathcal{O}(\mathrm{iters} \cdot n^3)$)",
             fontsize=12)
ax.grid(True, alpha=0.3, which="both")
ax.legend(fontsize=9, loc="best", ncol=2, framealpha=0.95)
plt.tight_layout()
plt.savefig(OUT_DIR / "runtime.png", dpi=140, bbox_inches="tight")
plt.show()

## Cell 8 — Discussion

**What the scaling figure shows.**

For balanced binary trees with $S_{ij} = \alpha^{d(i,j)}$, the matrix is
*exactly* low-rank in the structural sense — the eigenspectrum is determined
by the tree-distance structure and decays sharply with $\alpha$. This is the
ideal Candès-Recht setting. Any NNM advantage over IPW should be most visible
here. Compare the empirical $p^*$ values in cell 4: a widening NNM/IPW gap
as $n$ grows would match $p^*_{\text{NNM}} \sim r\log^2 n / n$.

**Practical takeaway.**

- Where IPW saturates first as $p$ grows, **IPW is the right default**
  (simpler, $\sim 10^2$–$10^3 \times$ faster — cell 7).
- Where NNM saturates first, the convex completion is buying you a real
  sample-complexity edge — useful only at the small-$p$ tail where the
  sample budget is the binding constraint.

**Caveats.**

- The $r\log^2 n / n$ overlay uses the simplest form of the theorem
  ($r=2$, no $\mu^2$ coherence factor); the constant in the theoretical
  bound is loose.
- $\alpha = 0.9$ here matches figure 1. Lower $\alpha$ would give an even
  more sharply low-rank $S$ (NNM advantage should grow); higher $\alpha$
  closer to 1 makes $S$ near-flat (rank ~1).
- IALM is not used — we use Soft-Impute, the standard NNM solver for
  matrix completion under uniform sampling. See `nnm_from_mask` docstring.